# B4 · MiniMax-H3 (비디오 + 오디오 joint 3D DiT) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/B4_minimax_h3.ipynb)

원본: DiffSynth-Studio `diffsynth/models/minimax_h3_dit.py` (`MiniMaxH3DiT`, commit `4a2d8af` 고정).
diffusers에는 아직 MiniMax-H3가 없어서, DiffSynth 전체 대신 이 파일과 이 파일이 import하는 `core/attention`, `core/gradient`
두 파일만 받아 묶어 씁니다 (attention은 torch SDPA 경로로 고정). 실제 체크포인트(`MiniMaxAI/MiniMax-H3` `FL2VA/transformer`)와
가중치 이름이 같습니다.

내 구현: `miniarch/models/minimax_h3.py`. 파라미터 이름은 원본과 같습니다.

부품
- **한 줄 packed 시퀀스** (batch 1): `[text | 조건 keyframe | audio | video | pad]`, 길이는 64의 배수로 padding
- 토큰화: video latent (1,2,2) patch → `video_patch_proj`, audio latent 채널별 시간 행 → `audio_patch_proj`,
  text → `condition_proj` → **token refiner** (RoPE·변조 없는 2층 transformer)
- **3축 RoPE** (t, h, w 실수 좌표, 축마다 주파수 16개): head_dim 128 중 앞 96 채널만 회전
- 블록 50개는 **모든 modality가 같은 가중치** (단일 스트림). attention은 실제 길이 / pad 두 구간으로 나눈 양방향
- **modality별 adaLN**: timestep 임베딩 하나 → (video / text / audio) × (shift, scale, gate) × (attn, mlp).
  토큰마다 자기 timestep과 modality 행을 고름. 조건 프레임은 t = 0.999, 편집(retake)에서 고정한 칸은 t = 1
- 출력층: modality 공통 shift/scale → `video_out` / `audio_out`

축소 config (모든 경로가 한 번씩 쓰이도록)
- hidden 64, head 4×16, 블록 2, refiner 2층. rope 주파수 축마다 2개 → 회전 12채널 + 회전 안 하는 4채널 (실제 96 / 128처럼)
- video latent [1, 4, T=3, H=4, W=6] → 18행, keyframe 조건 2장(첫/마지막 프레임 위치) → 12행, audio 2채널 × 5 → 10행,
  text 7토큰(그중 하나는 modality tag 0) → 실제 47 + pad 17 = 64
- timestep 4종(video 0.3 / audio 0.6 / 조건 0.999 / 편집 고정 1.0), 출력 mask(`skip_mask_out_condition=False`), ControlNet hint 1개
- 출력은 video 행과 audio 행을 모두 이어 붙여 비교합니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. 단계 전용 자체 점검 → 5. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
if not os.environ.get('MINIARCH_SKIP_PIP'):   # 전체 실행 노트북이 한 번에 설치
    !pip -q install -U 'einops'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("B4")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('MiniMaxAI/MiniMax-H3', '^(blocks\\.0\\.|token_refiner\\.blocks\\.0\\.|final_layer\\.|time_embedder\\.|rope\\.|\\w+_proj\\.)', subfolder='FL2VA/transformer')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## 단계 전용 자체 점검

### attention 구간 · 확률 합 · modality 간 주의

- `attn_probs`: [1, H, S, S]. 행 합 = 1, 실제 토큰 ↔ pad 사이 확률은 정확히 0 이어야 합니다 (구간 나눔).
- 0번 블록에서 query modality별로 key modality(text / video / audio)에 준 평균 확률 질량을 출력합니다.
- `mod_probe`: 서로 다른 변조 벡터 수 = 서로 다른 (timestep, modality) 쌍 수.

In [ ]:
from miniarch.models import minimax_h3 as H
ref, inp = S.ref()
m = H.verify_pair(ref)["mine"]
V.copy_weights(ref, m)
m = m.to(dev).eval()
mi = H.mine_inputs({k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in inp.items()})
names = [n for n, _ in m.named_modules() if n.endswith(("attn_probs", "mod_probe"))]
with V.capture(m, names) as st, torch.no_grad():
    m(**mi)
S_ = inp["x"].shape[1]
used = int(inp["packed_seq_params"]["cu_seqlens_q"][1])
seg = (torch.arange(S_, device=dev) >= used).long()
cross = seg[:, None] != seg[None, :]
for n in names:
    if n.endswith("attn_probs"):
        p = st[n]
        err = (p.sum(-1) - 1).abs().max().item()
        report(f"{n} 행 합 = 1", err < 1e-5, err)
        if p.shape[-1] == S_:   # 본 블록만 (refiner 는 text 길이 L × L, pad 없음)
            leak = p[:, :, cross].abs().max().item()
            report(f"{n} 실제 ↔ pad 확률 = 0", leak == 0, leak)
tags = inp["token_tags"].to(dev)
p0 = st["blocks.0.attn.attn_probs"][0].mean(0)
names_ = {1: "text", 0: "video", 2: "audio"}
for a in (1, 0, 2):
    row = {names_[b]: round(p0[tags == a][:, tags == b].sum(-1).mean().item(), 3) for b in (1, 0, 2)}
    print(f"query {names_[a]:<5} → {row}")
mp = st["blocks.0.mod_probe"]
key = inp["inverse_indices"].to(dev) * 3 + tags.clamp(min=0)
nd = torch.unique(mp.permute(1, 0, 2).reshape(S_, -1), dim=0).shape[0]
print("timestep 고유값", inp["unique_timesteps"].tolist(), " 변조 벡터 종류", nd)
report("변조 벡터 수 = (timestep, modality) 쌍 수", nd == key.unique().numel(), 0.0)

### padding 길이 무관 · t2va 경로 · 빈 text(CFG null) 경로

- 조건·편집 없는 기본 t2va 입력을 padding 단위 64 / 128 로 만들어, 원본 출력이 같은지 (pad 구간이 실제 토큰에 안 섞이는지) 봅니다.
- 같은 입력에서 내 구현 == 원본, 그리고 text 길이 0 (null text) 입력에서도 내 구현 == 원본인지 확인합니다.
- CFG: 길이가 다른 negative prompt로 한 번 더 돌려 `uncond + s·(cond − uncond)` 를 만들어 봅니다.

In [ ]:
from miniarch.models import minimax_h3 as H
ref, _ = S.ref()
m = H.verify_pair(ref)["mine"]
V.copy_weights(ref, m)
ref, m = ref.to(dev).eval(), m.to(dev).eval()
g = torch.Generator().manual_seed(5)
vl, al = torch.randn(1, 4, 2, 4, 4, generator=g).to(dev), torch.randn(2, 8, 3, generator=g).to(dev)
pe, neg = torch.randn(5, 40, generator=g).to(dev), torch.randn(3, 40, generator=g).to(dev)

def run(text, align=64, tv=500, ta=500):
    d = H.build_inputs(vl, al, text, torch.ones(text.shape[0], dtype=torch.long), tv, ta, seq_align=align)
    with torch.no_grad():
        return H.joined(ref(**d)), H.joined(m(**H.mine_inputs(d)))

r64, m64 = run(pe, 64)
r128, m128 = run(pe, 128)
e = (r64 - r128).abs().max().item()
report("padding 64 vs 128 원본 출력 동일", e < 5e-5, e)   # 행 수가 달라 GEMM 커널이 바뀔 수 있음
e = max((r64 - m64).abs().max().item(), (r128 - m128).abs().max().item())
report("t2va 경로 내 구현 == 원본", e < 1e-4, e)
r0, m0 = run(pe[:0])
e = (r0 - m0).abs().max().item()
report("빈 text 경로 내 구현 == 원본", e < 1e-4, e)
rn, mn = run(neg)
e = (rn - mn).abs().max().item()
report("negative prompt(길이 3) 내 구현 == 원본", e < 1e-4, e)
for s in (1.0, 5.0):
    cfg = mn + s * (m64 - mn)
    print(f"CFG s={s}: |guided| 평균 {cfg.abs().mean():.4f}  (cond 평균 {m64.abs().mean():.4f})")

### text 순서 · timestep 민감도

- token refiner는 위치 정보가 없어 text 순서를 바꾸면 출력도 같은 순서로 바뀌기만 해야 합니다 (permutation equivariant).
- 본 블록에서는 text도 t축 RoPE 위치(0..L-1)를 받으므로 순서를 바꾸면 video/audio 출력이 달라져야 합니다.
- audio timestep만 바꿔도 joint attention 때문에 video 출력이 달라져야 합니다.

In [ ]:
from miniarch.models import minimax_h3 as H
ref, _ = S.ref()
m = H.verify_pair(ref)["mine"]
V.copy_weights(ref, m)
m = m.to(dev).eval()
g = torch.Generator().manual_seed(6)
vl, al = torch.randn(1, 4, 2, 4, 4, generator=g).to(dev), torch.randn(2, 8, 3, generator=g).to(dev)
pe = torch.randn(6, 40, generator=g).to(dev)
perm = torch.randperm(6, generator=g).to(dev)
L_ = pe.shape[0]
with torch.no_grad():
    mask = H.segment_mask(torch.tensor([0, L_, L_]), L_, dev)
    z = m.condition_proj(pe)
    e = (m.token_refiner(z, mask)[perm] - m.token_refiner(z[perm], mask)).abs().max().item()
report("refiner: text 순서 equivariant", e < 1e-5, e)

def video_out(text, tv=500, ta=500):
    d = H.build_inputs(vl, al, text, torch.ones(text.shape[0], dtype=torch.long), tv, ta)
    with torch.no_grad():
        return m(**H.mine_inputs(d))[0]

base = video_out(pe)
d1 = (video_out(pe[perm]) - base).abs().max().item()
report("본 블록: text 순서가 video 출력을 바꿈 (RoPE)", d1 > 1e-4, d1)
d2 = (video_out(pe, ta=900) - base).abs().max().item()
report("audio timestep만 바꿔도 video 출력이 바뀜", d2 > 1e-4, d2)
d3 = (video_out(pe, tv=900) - base).abs().max().item()
report("video timestep 민감도", d3 > 1e-4, d3)

### ablation 스위치

나중에 부품별 비교 실험에 쓸 스위치를 하나씩 끄고 forward가 정상(유한값)이고 출력이 실제로 바뀌는지 확인합니다.

In [ ]:
import dataclasses
from miniarch.models import minimax_h3 as H
ref, inp = S.ref()
base = H.config_from_module(ref)
m0 = H.MiniMaxH3(base)
V.copy_weights(ref, m0)
m0 = m0.to(dev).eval()
mi = H.mine_inputs({k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in inp.items()})
with torch.no_grad():
    y0 = H.joined(m0(**mi))
for sw in ["use_rope", "use_qk_norm", "use_token_refiner", "modality_adaln", "use_gate"]:
    m = H.MiniMaxH3(dataclasses.replace(base, **{sw: False})).to(dev).eval()
    m.load_state_dict(m0.state_dict())
    with torch.no_grad():
        y = H.joined(m(**mi))
    d = (y - y0).abs().max().item()
    report(f"{sw}=False forward (유한·출력 변화)", bool(torch.isfinite(y).all()) and d > 1e-5, d)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/B4_minimax_h3.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_B4_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")